# TBX11K — Object Detection Pipeline (Colab)

## 4 Architectures: FCOS, EfficientDet-D2, RetinaNet, DETR

**Setup**:
1. Upload `OBJECT_DETECTION.zip` to Colab's file panel (Files tab → upload icon)
2. Place TBX11K `Images/` folder in Google Drive at `MyDrive/tbx11k/Images`
3. Run Cell 1 first, then any model cell individually

In [ ]:
# Cell 1: Setup + Data + EDA
import os
from google.colab import drive

drive.mount('/content/drive')

# Unzip uploaded project code
if not os.path.exists('/content/OBJECT_DETECTION'):
    !unzip -q /content/OBJECT_DETECTION.zip -d /content/
%cd /content/OBJECT_DETECTION

# Copy TBX11K dataset from Drive
if not os.path.exists('Images/train'):
    !cp -r /content/drive/MyDrive/tbx11k/Images Images

# Install dependencies
!pip install -q torch>=2.0.0 torchvision>=0.15.0 effdet timm pycocotools pillow tqdm matplotlib seaborn
import torch
print(f'PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Convert annotations to COCO format
!python convert.py --data-root Images --output-dir dataset

# Exploratory Data Analysis
!python eda.py --data-root Images

# Display EDA plots
from IPython.display import Image as DImg, display
for fname in ['tag_distribution.png', 'bbox_class_distribution.png', 'bbox_spatial_heatmap.png',
              'bbox_size_distribution.png', 'bbox_aspect_ratio.png', 'boxes_per_image.png', 'sample_grid.png']:
    path = f'results/eda/{fname}'
    if os.path.exists(path):
        display(DImg(path))

In [ ]:
# Cell 2: FCOS (Train + Evaluate + XAI)
%cd /content/OBJECT_DETECTION
!python train_fcos.py 2>&1 | tee results/fcos/training_log.txt
print('FCOS complete.')

In [ ]:
# Cell 3: EfficientDet-D2 (Train + Evaluate + XAI)
%cd /content/OBJECT_DETECTION
!python train_efficientdet.py 2>&1 | tee results/efficientdet/training_log.txt
print('EfficientDet-D2 complete.')

In [ ]:
# Cell 4: RetinaNet (Train + Evaluate + XAI)
%cd /content/OBJECT_DETECTION
!python train_retinanet.py 2>&1 | tee results/retinanet/training_log.txt
print('RetinaNet complete.')

In [ ]:
# Cell 5: DETR (Train + Evaluate + XAI)
%cd /content/OBJECT_DETECTION
!python train_detr.py 2>&1 | tee results/detr/training_log.txt
print('DETR complete.')

In [ ]:
# Cell 6: Model Comparison
%cd /content/OBJECT_DETECTION
import os, json
import pandas as pd
import matplotlib.pyplot as plt

models = ['fcos', 'efficientdet', 'retinanet', 'detr']
model_labels = ['FCOS', 'EfficientDet-D2', 'RetinaNet', 'DETR']

all_metrics = {}
for model_name in models:
    path = f'results/{model_name}/metrics.json'
    if os.path.exists(path):
        with open(path) as f:
            all_metrics[model_name] = json.load(f)

df = pd.DataFrame(all_metrics).T
df.index = model_labels
print('\n=== MODEL COMPARISON ===')
print(df.round(4).to_string())
df.round(4).to_csv('results/comparison.csv')
print('\nSaved to results/comparison.csv')

# Bar plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if 'mAP@0.5:0.95' in df.columns:
    df['mAP@0.5:0.95'].plot(kind='bar', ax=axes[0], color=['#4C72B0', '#DD8452', '#55A868', '#C44E52'])
    axes[0].set_ylabel('mAP@0.5:0.95')
    axes[0].set_title('Detection Performance')
    axes[0].set_ylim(0, max(df['mAP@0.5:0.95']) * 1.2)
    axes[0].grid(axis='y', alpha=0.3)

if 'AP_ActiveTB' in df.columns and 'AP_ObsoleteTB' in df.columns:
    df[['AP_ActiveTB', 'AP_ObsoleteTB']].plot(kind='bar', ax=axes[1])
    axes[1].set_ylabel('AP')
    axes[1].set_title('Per-Class Average Precision')
    axes[1].grid(axis='y', alpha=0.3)
    axes[1].legend(['ActiveTB', 'ObsoleteTB'])

plt.tight_layout()
plt.savefig('results/comparison_barplot.png', dpi=150)
plt.show()
print('Comparison plot saved to results/comparison_barplot.png')

In [ ]:
# Cell 7: View Confusion Matrices & XAI
%cd /content/OBJECT_DETECTION
from IPython.display import Image as DImg, display

models = ['fcos', 'efficientdet', 'retinanet', 'detr']
model_labels = ['FCOS', 'EfficientDet-D2', 'RetinaNet', 'DETR']

# Confusion matrices
print('=== CONFUSION MATRICES ===')
for model_name, label in zip(models, model_labels):
    cm_path = f'results/{model_name}/confusion_matrix.png'
    if os.path.exists(cm_path):
        print(f'\n{label}:')
        display(DImg(cm_path))

# XAI explanations (first 2 samples per model)
print('\n=== XAI EXPLANATIONS ===')
for model_name, label in zip(models, model_labels):
    explain_dir = f'results/{model_name}/explain'
    if os.path.isdir(explain_dir):
        print(f'\n{label}:')
        samples = sorted(os.listdir(explain_dir))[:2]
        for s in samples:
            display(DImg(os.path.join(explain_dir, s)))

---
## Summary

| Model | mAP@0.5:0.95 | mAP@0.5 | AP_ActiveTB | AP_ObsoleteTB |
|-------|-------------|---------|-------------|--------------|
| FCOS | ... | ... | ... | ... |
| EfficientDet-D2 | ... | ... | ... | ... |
| RetinaNet | ... | ... | ... | ... |
| DETR | ... | ... | ... | ... |

**To save results to Drive**: `!cp -r results /content/drive/MyDrive/tbx11k_results/`